# llm_env

> Text foraging for Phase 10: the Phase 8 benchmark world with named locations, rendered as text for a language-model agent, the two fixed action schemas, parsing (anything unusable becomes a counted "stay"), and voluntary reports to the Phase 7 channels.

In [ ]:
#| default_exp llm_env

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import math

import numpy as np

from explRL.harness import Blackboard, Lead, MessageBus

_STR, _NUM = {'type': 'string'}, {'type': 'number'}
FORAGE_SCHEMA = {'type': 'object', 'additionalProperties': False,
                 'required': ['action', 'target', 'reportLocation', 'reportValue', 'notes'],
                 'properties': {'action': {'type': 'string', 'enum': ['stay', 'go']}, 'target': _STR,
                                'reportLocation': _STR, 'reportValue': _NUM, 'notes': _STR}}
BOARD_SCHEMA = {'type': 'object', 'additionalProperties': False,
                'required': ['action', 'target', 'answer', 'claimTask', 'claimKind', 'notes'],
                'properties': {'action': {'type': 'string', 'enum': ['open', 'submit', 'wait']}, 'target': _STR,
                               'answer': {'type': 'integer'}, 'claimTask': _STR,
                               'claimKind': {'type': 'string', 'enum': ['none', 'working', 'solved']}, 'notes': _STR}}
_CONS, _VOW = 'bcdfghklmnprstvz', 'aeiou'


def locationNames(L, seed):
    "`L` distinct two-syllable names (consonant-vowel pairs, capitalised), fixed by the seed."
    syl = [c + v for c in _CONS for v in _VOW]
    pairs = np.random.default_rng(seed).choice(len(syl) ** 2, L, replace=False)
    return [(syl[p // len(syl)] + syl[p % len(syl)]).capitalize() for p in pairs]


def _num(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool) and math.isfinite(x)


class TextForagingEnv:
    "The Phase 8 benchmark world in words: named locations, text observations, parsed actions and voluntary reports."
    task, timeUnit, SCHEMA, unit = 'forage', 's', FORAGE_SCHEMA, 'food'

    def __init__(self, world, names):
        self.world, self.names = world, list(names)
        self.index = {n.lower(): j for j, n in enumerate(self.names)}
        self.shown, self.home = {}, {}          # home: agent -> home locations, listed first (squads)

    def rules(self):
        w = self.world
        ring = ', '.join(self.names)
        return (f"You are one of a team of foragers in a world of {w.L} locations arranged in a ring, in this order "
                f"(the last is next to the first): {ring}.\n\n"
                f"Food: {w.nPatches} of the locations hold a food patch at any time; the others hold nothing. While you "
                f"stay at a patch you collect food at up to {w.c:g} per second, but a patch has a limited supply rate "
                f"(between {min(w.intake):g} and {max(w.intake):g} per second, different for each patch) that is shared "
                "equally among everyone staying there, so crowding a patch lowers each forager's take. What you observe "
                f"after a stay is noisy (about {w.noise:.0%} noise). Every {w.Tmove:g} seconds all patches move at once "
                "to new random locations, so what was good before may be empty now.\n\n"
                f"Danger: {w.nDanger} locations are dangerous. Staying at one costs {w.dangerCost:g} per second (a "
                "negative reward). Danger locations never move and never hold food.\n\n"
                f"Time: staying anywhere takes {w.tStay:g} seconds. Going to another location takes the travel time shown, which "
                f"grows with the distance along the ring ({w.layout.T[0, 1]:g} seconds per step). Each of your turns is "
                "one action: stay where you are (and collect), or go to a named location (and collect nothing on the "
                "way). Your earnings are what you collect minus danger costs.\n\n"
                "Reports: after any turn you may report one location and the reward per second you believe it gives "
                "(leave reportLocation empty for no report). Depending on the team's set-up, reports go to a shared "
                "board that every teammate sees (averaged and fading with age), to two random teammates as a message, "
                "or to a coordinator who suggests a location to each forager. Reports are voluntary. Teammates' reports "
                "may be outdated or wrong, and you are free to ignore any suggestion.\n\n"
                "What you see each turn: the time, where you are, what your last action earned, the travel time to "
                "every other location (nearest first), and whatever your team's set-up shows: the board's averaged "
                "reports with their fading weight, the messages you received since your last turn, or the "
                "coordinator's suggestion. Below that come your own past turns (or your latest notes), so you can "
                "remember where you found food, where you were hurt, and when the patches last moved.\n\n"
                f"Example: you stayed at a location and earned {0.8 * w.tStay:g} over the {w.tStay:g} seconds (0.8 per "
                "second), so it holds a patch; if you report it (as 0.8), teammates may come and share it, which lowers "
                f"your own take if the patch is small. If you earned {-w.dangerCost * w.tStay:g}, it is a danger "
                "location: leave it. If you earned 0, there is nothing there (now).\n\n"
                "Reply with JSON only: action is 'stay' or 'go'; target is the location name for 'go' (empty for "
                "'stay'); reportLocation and reportValue give your report (empty name for none); notes is a short "
                "note to yourself (what you know, what you plan), shown back to you later. For example: "
                '{"action": "go", "target": "' + self.names[1] + '", "reportLocation": "", "reportValue": 0, '
                '"notes": "nothing here"}.')

    def observe(self, i, t, channel=None):
        w, nm = self.world, self.names
        w.advance(t)
        o = w.observe(i)
        loc, travel = o['location'], o['travel']
        home = self.home.get(i, ())
        order = sorted(range(w.L), key=lambda j: (j not in home, travel[j], j))
        lines = [f"Time {t:.1f} s. You are at {nm[loc]}. Your last action earned {o['lastReward']:.2f}.",
                 'Travel times from ' + nm[loc] + ': ' + ', '.join(f'{nm[j]} {travel[j]:.1f} s' for j in order if j != loc)]
        shown = set()
        if isinstance(channel, Blackboard):
            S, C = channel.view(i, t)
            js = sorted(np.flatnonzero(C > 0), key=lambda j: (-S[j] / C[j], j))
            shown = {int(j) for j in js}
            lines.append('Board (teammates\' reports, mean reward per second, weight fades with age): ' +
                         ('; '.join(f'{nm[j]} {S[j] / C[j]:.2f} (weight {C[j]:.1f})' for j in js) or 'empty'))
        elif isinstance(channel, MessageBus):
            msgs = channel.take(i)
            shown = {j for _, j, _, _ in msgs}
            self._heard = getattr(self, '_heard', {})
            self._heard[i] = [(nm[j], v) for _, j, v, _ in msgs]
            lines.append('Messages: ' + ('; '.join(f'agent {s} says {nm[j]} {v:.2f} ({t - ts:.1f} s ago)'
                                                    for s, j, v, ts in msgs) or 'none'))
        elif isinstance(channel, Lead):
            j = channel.assign(i, loc, travel, t)
            shown = {j}
            lines.append(f'The coordinator suggests: {nm[j]}.')
        self.shown[i] = shown
        return '\n'.join(lines)

    def parse(self, parsed, i):
        "-> (target index, report (location, value) or None, valid); anything unusable is 'stay', invalid."
        loc = self.world.loc[i]
        if not isinstance(parsed, dict):
            return loc, None, False
        rep, rl = None, str(parsed.get('reportLocation', '')).strip().lower()
        if rl in self.index and _num(parsed.get('reportValue')):
            rep = (self.index[rl], float(parsed['reportValue']))
        if parsed.get('action') == 'stay':
            return loc, rep, True
        j = self.index.get(str(parsed.get('target', '')).strip().lower()) if parsed.get('action') == 'go' else None
        return (loc, rep, False) if j is None else (j, rep, True)

    def post(self, i, report, t, channel, info):
        if report is None or channel is None:
            return False
        j, v = report
        if isinstance(channel, Blackboard):
            channel.write(i, j, v, t)
        elif isinstance(channel, MessageBus):
            return bool(channel.due(i, t) and channel.send(self.world, i, j, v, t))
        elif isinstance(channel, Lead):
            channel.report(i, j, v, info.get('nHere', 1), t)
        return True

    def explored(self, i, action, info):
        "A move to a location no teammate's report or suggestion pointed to this turn."
        return info['kind'] == 'travel' and action not in self.shown.get(i, ())

    def summary(self, i, t, action, reward, report, info):
        s = f"t={t:.1f} s: {'stayed at' if info['kind'] == 'stay' else 'went to'} {self.names[self.world.loc[i]]}, " \
            f"earned {reward:.2f}"
        if report:
            s += f'; you reported {self.names[report[0]]} {report[1]:.2f}'
        heard = getattr(self, '_heard', {}).get(i)
        if heard:
            s += '; messages: ' + ', '.join(f'{n} {v:.2f}' for n, v in heard)
        return s + '.'

## Tests

In [ ]:
import json
import numpy as np
from explRL.llm_env import *
from explRL.harness import Blackboard, Lead, MessageBus
from explRL.harness_runs import benchWorld
from explRL.multi import spawnRngs

names = locationNames(64, 7)
assert names == locationNames(64, 7) and len(set(names)) == 64 and names != locationNames(64, 8)
assert all(n.isalpha() and n[0].isupper() and len(n) == 4 for n in names)

w = benchWorld(); w.reset(3, 2)
env = TextForagingEnv(w, names)
# the rules are well above the 512-token prompt-cache minimum (at a conservative 5 characters per token)
assert len(env.rules()) / 5 > 512, len(env.rules())
assert 'staying anywhere takes 1 seconds' in env.rules()
w5 = benchWorld(tStay=5.0); w5.reset(3, 2)
assert 'staying anywhere takes 5 seconds' in TextForagingEnv(w5, names).rules() and 'earned 4 over the 5 seconds' in TextForagingEnv(w5, names).rules()
obs = env.observe(0, 0.0)
loc = w.loc[0]
assert f'You are at {names[loc]}' in obs
for j in range(64):     # every other location is listed by name with its travel time
    assert j == loc or f'{names[j]} {w.layout.T[loc, j]:.1f} s' in obs

# parsing: names are case-insensitive; anything unusable is 'stay' and invalid
other = names[(loc + 3) % 64]
assert env.parse(dict(action='go', target=other.upper(), reportLocation='', reportValue=0, notes=''), 0) == \
    ((loc + 3) % 64, None, True)
assert env.parse(dict(action='stay', target='', reportLocation=other, reportValue=0.7, notes=''), 0) == \
    (loc, ((loc + 3) % 64, 0.7), True)
for bad in [None, 'text', dict(action='go', target='Nowhere'), dict(action='jump'),
            dict(action='stay', reportLocation=other, reportValue=float('nan'))]:
    t, rep, ok = env.parse(bad, 0)
    assert t == loc and rep is None and (not ok or bad.get('action') == 'stay')

# a scripted team through the text interface leaves the same ledger as the same moves made directly
def moves(k):
    rng = np.random.default_rng(k)
    return [('go', names[int(rng.integers(64))]) if rng.random() < 0.3 else ('stay', '') for _ in range(40)]
def play(text):
    w = benchWorld(); w.reset(5, 2); e = TextForagingEnv(w, names); q, n = [(0.0, 0), (0.0, 1)], [0, 0]
    while q:                                     # event order: earliest start first, ties to the lower index
        q.sort(); t, i = q.pop(0)
        a, tg = moves(i)[n[i]]
        if text:
            e.observe(i, t)
            target, _, _ = e.parse(dict(action=a, target=tg, reportLocation='', reportValue=0, notes=''), i)
        else:
            target = w.loc[i] if a == 'stay' else names.index(tg)
        dt = w.apply(i, target, t)[1]
        n[i] += 1
        if n[i] < 40:
            q.append((t + dt, i))
    return w.ledger
assert json.dumps(play(True), default=str) == json.dumps(play(False), default=str) and len(play(True)) == 80

# channels: reports reach the board, two teammates' inboxes, or the lead; the observation shows them
w = benchWorld(); w.reset(1, 3); env = TextForagingEnv(w, names)
bb = Blackboard(64, None, 0.01, 3)
assert env.post(0, (5, 0.9), 1.0, bb, {}) and 'Board' in env.observe(1, 2.0, bb) and f'{names[5]} 0.90' in env.observe(1, 2.0, bb)
assert env.shown[1] == {5} and 'empty' in env.observe(0, 2.0, bb)            # own reports are not shown back
env.home[2] = {40, 41}                                                         # squads: home locations listed first
listed = [x.split(' ')[0] for x in env.observe(2, 2.0).split('\n')[1].split(': ', 1)[1].split(', ')]
assert set(listed[:2]) == {names[40], names[41]} and len(listed) == 63
mb = MessageBus(3, 2, 10.0, np.inf, spawnRngs(9, 3))
assert env.post(0, (6, 1.1), 1.0, mb, {}) and not env.post(0, (6, 1.1), 2.0, mb, {})   # once per period
o1, o2 = env.observe(1, 3.0, mb), env.observe(2, 3.0, mb)
assert f'agent 0 says {names[6]} 1.10' in o1 and f'agent 0 says {names[6]} 1.10' in o2
assert 'messages:' in env.summary(1, 3.0, w.loc[1], 0.0, None, dict(kind='stay'))
lead = Lead(64, 3)
assert 'The coordinator suggests' in env.observe(0, 3.0, lead) and lead.assigned[0] is not None
env.post(0, (7, 1.0), 3.0, lead, dict(nHere=1))
assert lead.table.view(-1, 3.0)[1][7] > 0
# exploration: a move to a location nobody pointed to
env.observe(1, 4.0, bb)
assert env.explored(1, 9, dict(kind='travel')) and not env.explored(1, 5, dict(kind='travel'))
assert not env.explored(1, 9, dict(kind='stay'))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()